In [1]:
# For data manipulation
import numpy as np
import pandas as pd

# For data visualization
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)

In [ ]:
df0 = pd.read_csv('data_for_modeling_5.csv')
df0.rename(columns={'Unnamed: 0': 'company_id'}, inplace=True)
df0.set_index('company_id', inplace=True)

In [ ]:
df1 = df0.drop(columns=['Company Name', 'Symbol','Change 1W','Change 1M', 'Change 6M', 'Change 1Y', 'EBITDA'])

In [4]:
df1['IPO Date'] = pd.to_datetime(df1['IPO Date'])
df1[['IPO Date']].info()

<class 'pandas.core.frame.DataFrame'>
Index: 2134 entries, 0 to 2402
Data columns (total 1 columns):
 #   Column    Non-Null Count  Dtype         
---  ------    --------------  -----         
 0   IPO Date  2134 non-null   datetime64[ns]
dtypes: datetime64[ns](1)
memory usage: 33.3 KB


In [5]:
df_test = df1[df1['IPO Date'] > pd.Timestamp('2025-01-01')]
df_train = df1[~df1.index.isin(df_test.index)]

In [6]:
df_train.drop(columns=['IPO Date', ], inplace=True)
df_test.drop(columns=['IPO Date','high_risk'], inplace=True)


/var/folders/0y/v4q12cx949g1p48j2rym4v600000gn/T/ipykernel_1556/3697228244.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_train.drop(columns=['IPO Date', ], inplace=True)
/var/folders/0y/v4q12cx949g1p48j2rym4v600000gn/T/ipykernel_1556/3697228244.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_test.drop(columns=['IPO Date','high_risk'], inplace=True)


In [9]:
%%time
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from sklearn.model_selection import GridSearchCV, cross_val_score, train_test_split
from sklearn.metrics import roc_auc_score

# Prepare train and test data
X = df_train.drop(columns=['high_risk'])
y = df_train['high_risk']
X_test = df_test.copy()

# Identify numerical and categorical features
numerical_cols = X.select_dtypes(include=['float64', 'int']).columns.tolist()
categorical_cols = ['Sector']

# Preprocessing pipelines
numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])
cat_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer([
    ('num', numeric_transformer, numerical_cols),
    ('cat', cat_transformer, categorical_cols)
])

# Define model pipelines
pipe_rf = Pipeline([
    ('pre', preprocessor),
    ('clf', RandomForestClassifier(random_state=42))
])
pipe_xgb = Pipeline([
    ('pre', preprocessor),
    ('clf', XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42))
])
pipe_cat = Pipeline([
    ('pre', preprocessor),
    ('clf', CatBoostClassifier(verbose=0, random_state=42))
])

# Parameter grids for GridSearchCV
param_grid_rf = {'clf__n_estimators': [100, 200], 'clf__max_depth': [None, 10]}
param_grid_xgb = {'clf__n_estimators': [100, 200], 'clf__max_depth': [3, 6], 'clf__learning_rate': [0.01, 0.1]}
param_grid_cat = {'clf__iterations': [100, 200], 'clf__depth': [3, 6], 'clf__learning_rate': [0.01, 0.1]}

# Train-test split for validation
X_train, X_val, y_train, y_val = train_test_split(X, y, stratify=y, test_size=0.2, random_state=42)

# Grid search with cross-validation
gs_rf = GridSearchCV(pipe_rf, param_grid_rf, cv=5, scoring='roc_auc', n_jobs=-1)
gs_xgb = GridSearchCV(pipe_xgb, param_grid_xgb, cv=5, scoring='roc_auc', n_jobs=-1)
gs_cat = GridSearchCV(pipe_cat, param_grid_cat, cv=5, scoring='roc_auc', n_jobs=-1)

gs_rf.fit(X_train, y_train)
gs_xgb.fit(X_train, y_train)
gs_cat.fit(X_train, y_train)

# Best estimators
best_rf = gs_rf.best_estimator_
best_xgb = gs_xgb.best_estimator_
best_cat = gs_cat.best_estimator_

# Evaluate on validation set
def eval_model(model, X_val, y_val):
    proba = model.predict_proba(X_val)[:, 1]
    return roc_auc_score(y_val, proba)

print('RF AUC:', eval_model(best_rf, X_val, y_val))
print('XGB AUC:', eval_model(best_xgb, X_val, y_val))
print('CAT AUC:', eval_model(best_cat, X_val, y_val))

# Voting classifier
voting = VotingClassifier(
    estimators=[('rf', best_rf), ('xgb', best_xgb), ('cat', best_cat)],
    voting='soft'
)
# Cross-validation for ensemble
cv_scores = cross_val_score(voting, X, y, cv=5, scoring='roc_auc')
print('Voting CV AUC scores:', cv_scores)
print('Mean CV AUC:', np.mean(cv_scores))

# Fit on full data and predict
voting.fit(X, y)
pred = voting.predict(X_test)
proba = voting.predict_proba(X_test)[:, 1]

df_test['high_risk_pred'] = pred
df_test['high_risk_proba'] = proba
print(df_test[['high_risk_pred', 'high_risk_proba']].head())

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/xgboost/core.py:158: UserWarning: [11:43:50] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/xgboost/core.py:158: UserWarning: [11:43:50] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/xgboost/core.py:158: UserWarning: [11:43:50] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/xgboost/core.py:158: UserWarning: [11:43:50] WARNING: /Users/runner/work/xgboost/xgboost/src

RF AUC: 0.8851438974512446
XGB AUC: 0.881752207092623
CAT AUC: 0.8956057658736097


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/xgboost/core.py:158: UserWarning: [11:43:58] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/xgboost/core.py:158: UserWarning: [11:43:59] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/xgboost/core.py:158: UserWarning: [11:44:00] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/xgboost/core.py:158: UserWarning: [11:44:01] WARNING: /Users/runner/work/xgboost/xgboost/src

Voting CV AUC scores: [0.85853409 0.88580193 0.90316672 0.91994128 0.87543912]
Mean CV AUC: 0.8885766287938466


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/xgboost/core.py:158: UserWarning: [11:44:03] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


            high_risk_pred  high_risk_proba
company_id                                 
2305                     1         0.770286
2306                     1         0.665035
2307                     1         0.777332
2308                     0         0.419882
2309                     1         0.632712
CPU times: user 15.8 s, sys: 2.95 s, total: 18.7 s
Wall time: 17.1 s


<timed exec>:95: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
<timed exec>:96: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy


In [ ]:
pd.set_option('display.max_rows', 100)
df_test[['high_risk_pred', 'high_risk_proba']].sort_values(by=['high_risk_pred', 'high_risk_proba'], ascending=False).head(10)

,high_risk_pred,high_risk_proba
company_id,,
2370,1,0.923470
2393,1,0.907315
2390,1,0.904520
2313,1,0.901384
2337,1,0.889776
2365,1,0.879916
2335,1,0.858238
2317,1,0.849190
2364,1,0.843392


In [16]:
df_result = df_test.join(df0[['Company Name']], how='left')
df_result[['Company Name', 'Sector', 'high_risk_pred', 'high_risk_proba']].sort_values(by=['high_risk_pred','high_risk_proba'], ascending=False)

,Company Name,Sector,high_risk_pred,high_risk_proba
company_id,,,,
2370,FBS Global Limited,Industrials,1,0.923470
2393,PicoCELA Inc.,Communication Services,1,0.907315
2390,Toppoint Holdings Inc.,Industrials,1,0.904520
2313,iOThree Limited,Technology,1,0.901384
2337,RedCloud Holdings plc,Industrials,1,0.889776
2365,Aureus Greenway Holdings Inc.,Consumer Discretionary,1,0.879916
2335,Baiya International Group Inc.,Industrials,1,0.858238
2317,Ruanyun Edai Technology Inc.,Diversified Consumer Services,1,0.849190
2364,"TEN Holdings, Inc.",Communication Services,1,0.843392


In [25]:
df_res = df_result.groupby('Sector')['high_risk_pred'].agg(['sum', 'count']).sort_values(by='sum', ascending=False).reset_index()

In [26]:
df_res['portion'] = round(df_res['sum']/df_res['count'], 2)
df_res

,Sector,sum,count,portion
0,Industrials,19,22,0.86
1,Technology,8,10,0.80
2,Healthcare,6,12,0.50
3,Communication Services,4,5,0.80
4,Consumer Discretionary,4,5,0.80
5,Consumer Staples,3,4,0.75
6,Financials,3,26,0.12
7,Energy,2,5,0.40
8,Diversified Consumer Services,1,1,1.00
9,Everbright Digital,1,1,1.00
